# LTX-2.3 22B Distilled 1.1 Q4 — Video Generator

Self-contained Kaggle notebook for Text-to-Video and Image-to-Video generation with support for both single generation and headless bulk mode.

GitHub: https://github.com/kcblak/LTX-2.3-22B-bulk-1.1-Q4-Video-Generator-by-blak

## Step 1: Environment Setup

In [ ]:
import os
import psutil
import subprocess

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["MALLOC_TRIM_THRESHOLD_"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("RAM:")
print(f"  Total: {psutil.virtual_memory().total / 1024**3:.2f} GB")
print(f"  Available: {psutil.virtual_memory().available / 1024**3:.2f} GB")

print("\nDisk:")
subprocess.run(["df", "-h", "/kaggle/working", "/kaggle/tmp"], check=False)

## Step 2: Clone Wan2GP & Install PyTorch

In [ ]:
import os
import subprocess

if subprocess.run("nvidia-smi", shell=True).returncode != 0:
    print("\n⚠️  No NVIDIA GPU detected. In Kaggle, go to Settings → Accelerator → GPU T4 x1, then restart the session.")

if not os.path.isdir("Wan2GP"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/DeepBeepMeep/Wan2GP.git"], check=True)
else:
    print("Wan2GP already cloned.")

subprocess.run([
    "pip", "install", "-q",
    "torch==2.3.1", "torchvision==0.18.1", "torchaudio==2.3.1",
    "--index-url", "https://download.pytorch.org/whl/cu121"
], check=True)

print("\n✅ PyTorch 2.3.1 installed. Now: Kernel → Restart Kernel, then Run All.")

## Step 3: Install Core Dependencies (after kernel restart)

In [ ]:
from pathlib import Path
import subprocess
import torch

DEPS = [
    "diffusers>=0.25.0", "transformers>=4.40.0", "accelerate>=0.30.0",
    "safetensors>=0.4.0", "Pillow>=10.0.0", "opencv-python>=4.9.0",
    "sentencepiece>=0.2.0", "peft>=0.8.0", "huggingface-hub>=0.20.0",
    "ffmpeg-python>=0.2.0", "google-api-python-client>=2.0.0",
]
for dep in DEPS:
    subprocess.run(["pip", "install", "--timeout", "120", "-q", dep], check=True)

subprocess.run(["pip", "install", "--timeout", "120", "-q", "mmgp", "gradio", "gguf", "soundfile"], check=True)

assert torch.__version__.startswith("2.3.1"), (
    f"torch was upgraded to {torch.__version__} — "
    f"this breaks the sm_60/P100 compatibility patches. "
    f"Re-run cell 2 (PyTorch install) and do NOT install any package that pulls a newer torch."
)
print(f"✅ torch version locked at {torch.__version__}")

## Step 4: Download Models

In [ ]:
import os
import shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

REPO = "Abiray/LTX-2.3-22B-DISTILLED-1.1-GGUF"
COMPANION_REPO = "DeepBeepMeep/LTX-2"
MODEL_DIR = Path("Wan2GP/models")
TMP_DIR = Path("/kaggle/tmp/models")
CACHE_DIR = Path("/kaggle/tmp/hf_cache")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

hf_token = os.environ.get("HF_TOKEN") or None
download_kwargs = {"token": hf_token} if hf_token else {}

def _download(repo_id, filename, dest):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        print(f"✓ Already exists: {dest}")
        return
    print(f"Downloading {repo_id}/{filename} -> {dest}")
    downloaded = hf_hub_download(
        repo_id=repo_id, filename=filename,
        local_dir=str(dest.parent), local_dir_use_symlinks=False,
        cache_dir=str(CACHE_DIR), **download_kwargs,
    )
    if Path(downloaded).resolve() != dest.resolve():
        shutil.move(str(downloaded), str(dest))

def _symlink_if_needed(target, link_name):
    target, link_name = Path(target), Path(link_name)
    if link_name.exists():
        print(f"✓ Already linked: {link_name}")
        return
    try:
        link_name.symlink_to(target)
        print(f"✓ Linked {link_name} -> {target}")
    except OSError:
        shutil.copy2(target, link_name)
        print(f"✓ Copied fallback: {link_name}")

_transformer_tmp = TMP_DIR / "ltx-2.3-22b-distilled-1.1-Q4_K_M.gguf"
_transformer_link = MODEL_DIR / "ltx-2.3-22b-distilled-1.1-Q4_K_M.gguf"
_download(REPO, "LTX-2.3-22B-distilled-1.1-Q4_K_M.gguf", _transformer_tmp)
_symlink_if_needed(_transformer_tmp, _transformer_link)

for f in ["ltx-2.3-22b-distilled-lora-384.safetensors", "ltx-2.3-22b_embeddings_connector.safetensors",
          "ltx-2.3-22b_text_embedding_projection.safetensors", "ltx-2.3-22b_vae.safetensors"]:
    tmp, link = TMP_DIR / f, MODEL_DIR / f
    _download(COMPANION_REPO, f, tmp)
    _symlink_if_needed(tmp, link)

for f in ["ltx-2.3-22b_audio_vae.safetensors", "ltx-2.3-22b_vocoder.safetensors",
          "ltx-2.3-spatial-upscaler-x2-1.1.safetensors", "ltx-2.3-temporal-upscaler-x2-1.0.safetensors"]:
    _download(COMPANION_REPO, f, MODEL_DIR / f)

gemma_dir = MODEL_DIR / "gemma-3-12b-it-qat-q4_0-unquantized"
for f in ["gemma-3-12b-it-qat-q4_0-unquantized_quanto_bf16_int8.safetensors", "added_tokens.json", "chat_template.json",
          "config_light.json", "generation_config.json", "preprocessor_config.json", "processor_config.json",
          "special_tokens_map.json", "tokenizer.json", "tokenizer.model", "tokenizer_config.json"]:
    _download(COMPANION_REPO, f"gemma-3-12b-it-qat-q4_0-unquantized/{f}", gemma_dir / f)

print("\nDisk after downloads:")
subprocess.run(["df", "-h", "/kaggle/working", "/kaggle/tmp"], check=False)

## Step 5: Load Google Drive Secret (for auto-backup)

In [ ]:
import os

GDRIVE_AVAILABLE = False
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    sa_json = secrets.get_secret("GDRIVE_SERVICE_ACCOUNT_JSON") or ""
    shared_drive_id = secrets.get_secret("GDRIVE_SHARED_DRIVE_ID") or ""
    os.environ["GDRIVE_SERVICE_ACCOUNT_JSON"] = sa_json
    os.environ["GDRIVE_SHARED_DRIVE_ID"] = shared_drive_id
    GDRIVE_AVAILABLE = bool(sa_json)
    print("✅ Google Drive secrets loaded from Kaggle Secrets." if GDRIVE_AVAILABLE else "⚠️ Google Drive secrets not configured; local videos will be retained.")
except Exception as e:
    os.environ.setdefault("GDRIVE_SERVICE_ACCOUNT_JSON", "")
    os.environ.setdefault("GDRIVE_SHARED_DRIVE_ID", "")
    print(f"⚠️ Google Drive secrets not configured or unavailable; local videos will be retained. ({e})")

## Step 6: Headless Bulk Mode OR Single Video Generation

Choose one option:
- **Headless**: Run bulk processing from dataset `kcblak/ltx-batch`
- **Single UI**: Launch Gradio interface for interactive generation

In [ ]:
# %%HEADLESS BULK MODE (recommended for batch processing)
# !pip install -q ffmpeg-python  # ensure available
# !git clone https://github.com/kcblak/LTX-2.3-22B-bulk-1.1-Q4-Video-Generator-by-blak ltx-core 2>/dev/null
# %cd ltx-core
# !python main.py --init --project MyLTXProject
# !python main.py --run --headless

print("Headless mode: Uncomment the lines above and run this cell to process jobs.csv from dataset kcblak/ltx-batch")

In [ ]:
# %%SINGLE VIDEO GENERATION - Gradio UI
%%writefile run_ltx_t2v.py
import gc, os, sys, json, random, threading, time, csv, io, zipfile, shutil
import numpy as np, psutil, soundfile as sf
from PIL import Image

WAN2GP_DIR = os.path.abspath("Wan2GP")
sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.5"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
assert torch.__version__.startswith("2.3.1"), f"torch=={torch.__version__}, expected 2.3.1"

_GPU_SM = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
_IS_SM60 = (_GPU_SM[0] == 6)
if _IS_SM60:
    os.environ["WGP_DTYPE"] = "fp16"
    print("  [GPU] sm_60 detected (P100) — FP16 mode enabled")
else:
    print(f"  [GPU] sm_{_GPU_SM[0]}{_GPU_SM[1]} detected")

import gradio as gr
from shared.utils.audio_video import save_video

GDRIVE_SA_JSON = os.environ.get("GDRIVE_SERVICE_ACCOUNT_JSON", "")
GDRIVE_SHARED_DRIVE_ID = os.environ.get("GDRIVE_SHARED_DRIVE_ID", "")
GDRIVE_AVAILABLE = False
_gdrive_service = None

def _init_gdrive():
    global _gdrive_service, GDRIVE_AVAILABLE
    if not GDRIVE_SA_JSON:
        print("  [GDrive] No service account configured")
        return None
    try:
        from google.oauth2 import service_account
        from googleapiclient.discovery import build
        info = json.loads(GDRIVE_SA_JSON)
        creds = service_account.Credentials.from_service_account_info(info, scopes=["https://www.googleapis.com/auth/drive"])
        _gdrive_service = build("drive", "v3", credentials=creds, cache_discovery=False)
        GDRIVE_AVAILABLE = True
        print("  [GDrive] ✅ Service account authenticated")
    except Exception as e:
        print(f"  [GDrive] ⚠️ Initialization failed: {e}")

def upload_video_to_gdrive(video_path, folder_id=None):
    if not GDRIVE_AVAILABLE or not video_path or not os.path.exists(video_path):
        return None
    try:
        from googleapiclient.http import MediaFileUpload
        filename = os.path.basename(video_path)
        metadata = {"name": filename}
        if folder_id:
            metadata["parents"] = [folder_id]
        media = MediaFileUpload(video_path, mimetype="video/mp4", resumable=True)
        kwargs = {"body": metadata, "media_body": media, "fields": "id"}
        if GDRIVE_SHARED_DRIVE_ID:
            kwargs["supportsAllDrives"] = True
        file = _gdrive_service.files().create(**kwargs).execute()
        return file.get("id")
    except Exception as e:
        print(f"  [GDrive] ❌ Upload failed: {e}")
        return None

def _get_or_create_gdrive_folder(folder_name="LTX_Videos_Backup"):
    if not GDRIVE_AVAILABLE:
        return None
    try:
        query = f"name='{folder_name}' and trashed=false and mimeType='application/vnd.google-apps.folder'"
        kwargs = {"q": query, "fields": "files(id, name)"}
        if GDRIVE_SHARED_DRIVE_ID:
            kwargs.update(corpora="drive", driveId=GDRIVE_SHARED_DRIVE_ID,
                       includeItemsFromAllDrives=True, supportsAllDrives=True)
        files = _gdrive_service.files().list(**kwargs).execute().get("files", [])
        if files:
            return files[0]["id"]
        metadata = {"name": folder_name, "mimeType": "application/vnd.google-apps.folder"}
        if GDRIVE_SHARED_DRIVE_ID:
            metadata["parents"] = [GDRIVE_SHARED_DRIVE_ID]
        kwargs2 = {"body": metadata, "fields": "id"}
        if GDRIVE_SHARED_DRIVE_ID:
            kwargs2["supportsAllDrives"] = True
        return _gdrive_service.files().create(**kwargs2).execute().get("id")
    except Exception as e:
        print(f"  [GDrive] ⚠️ Folder error: {e}")
        return None

_init_gdrive()
GDRIVE_FOLDER_ID = _get_or_create_gdrive_folder() if GDRIVE_AVAILABLE else None

def _register_gguf_handler():
    import shared.qtypes.gguf
    print("  [GGUF] ✅ Extension handler registered")

def _patch_ltx2_config_loading():
    import models.ltx2.ltx2 as ltx2_mod
    _original = ltx2_mod._load_config_from_checkpoint
    def _patched(path, fallback_config_path=None):
        from mmgp import quant_router
        if isinstance(path, (list, tuple)):
            path = path[0] if path else ""
        if not path:
            return {}
        try:
            _, metadata = quant_router.load_metadata_state_dict(path)
            config = ltx2_mod._normalize_config(metadata.get("config", {}))
            if config:
                return config
        except Exception:
            pass
        if fallback_config_path and os.path.isfile(fallback_config_path):
            try:
                with open(fallback_config_path) as f:
                    config = ltx2_mod._normalize_config(json.load(f))
                    if config:
                        return config
            except Exception:
                pass
        return {}
    ltx2_mod._load_config_from_checkpoint = _patched

print(f"GPU: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
sys.stdout.flush()

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)

print("\nLoading LTX-2.3 22B Distilled 1.1 (GGUF Q4_K_M)...")
sys.stdout.flush()

from mmgp import offload
from shared.utils import files_locator as fl
fl.set_checkpoints_paths(["models", "ckpts", "."])

from models.ltx2.ltx2_handler import family_handler
_register_gguf_handler()
_patch_ltx2_config_loading()

if _IS_SM60:
    import torch.nn.functional as _F
    from models.ltx2.ltx_core.model.audio_vae.causal_conv_2d import CausalConv2d as _CC2d
    def _cc2d_cpu_pad(self, x):
        if x.is_cuda:
            dev, dt = x.device, x.dtype
            x_cpu = x.detach().cpu().float()
            x_cpu = _F.pad(x_cpu, self.padding)
            w = self.conv.weight.detach().cpu().float()
            b = self.conv.bias.detach().cpu().float() if self.conv.bias else None
            out = _F.conv2d(x_cpu, w, b, self.conv.stride, self.conv.padding,
                          self.conv.dilation, self.conv.groups)
            return out.to(device=dev, dtype=dt)
        return _F.pad(x, self.padding); self.conv(x)
    _CC2d.forward = _cc2d_cpu_pad
    print("  [sm_60 Fix] ✅ CausalConv2d patched")

import glob
base_model_type = "ltx2_22B"
model_def = {"ltx2_pipeline": "distilled"}
extra = family_handler.query_model_def(base_model_type, model_def)
model_def.update(extra)

gemma_folder = "models/gemma-3-12b-it-qat-q4_0-unquantized"
gemma_files = sorted(glob.glob(os.path.join(gemma_folder, "*.safetensors"))) or []
text_encoder_file = next((f for f in gemma_files if "quanto" in f), gemma_files[0])
transformer_path = os.path.join("models", "ltx-2.3-22b-distilled-1.1-Q4_K_M.gguf")

ltx2_model, pipe = family_handler.load_model(
    model_filename=transformer_path, model_type="ltx2_22B_distilled", base_model_type=base_model_type,
    model_def=model_def, dtype=torch.float16, VAE_dtype=torch.float16, text_encoder_filename=text_encoder_file,
)

offload.profile(
    pipe, profile_no=4, quantizeTransformer=False, convertWeightsFloatTo=torch.float16,
    budgets={"transformer": 6000, "text_encoder": 1500, "video_encoder": 2000,
             "video_decoder": 3000, "audio_encoder": 1000, "audio_decoder": 1000,
             "vocoder": 500, "spatial_upsampler": 1500, "vae": 1000, "*": 1000},
)
offload.shared_state["_attention"] = "sdpa"
print("\n✅ Setup complete!")
sys.stdout.flush()

OUTPUT_DIR = "/kaggle/working/outputs"
_output_counter = 0

def list_outputs():
    if not os.path.isdir(OUTPUT_DIR):
        return []
    return sorted([f for f in os.listdir(OUTPUT_DIR) if f.lower().endswith(('.mp4', '.mkv', '.webm'))],
                  key=lambda x: os.path.getmtime(os.path.join(OUTPUT_DIR, x)), reverse=True)

def _get_next_output_path():
    global _output_counter
    _output_counter += 1
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    return os.path.join(OUTPUT_DIR, f"ltx_video_{_output_counter:04d}.mp4")

def get_resolution(base_res_str, aspect_ratio_str):
    base = {"1080p": 1088, "720p": 704, "540p": 544, "480p": 480}.get(base_res_str, 704)
    ratio = {"16:9 Landscape": 16/9, "4:3 Standard": 4/3, "1:1 Square": 1.0,
             "3:4 Portrait": 3/4, "9:16 Portrait": 9/16}.get(aspect_ratio_str, 16/9)
    if ratio >= 1.0:
        return (base // 32) * 32, int(base * ratio // 32) * 32
    return int(base // ratio // 32) * 32, (base // 32) * 32

@torch.inference_mode()
def Video_Generation(prompt, input_image_start, input_image_end, seed, duration_dropdown,
                      resolution_dropdown, aspect_ratio_dropdown, guide_scale=3.0,
                      num_steps=8, progress=gr.Progress()):
    duration_map = {"2 Seconds (49 frames)": 49, "3 Seconds (73 frames)": 73,
                    "5 Seconds (121 frames)": 121, "8 Seconds (193 frames)": 193,
                    "10 Seconds (241 frames)": 241, "15 Seconds (361 frames)": 361}
    num_frames = duration_map.get(duration_dropdown, 121)
    width, height = get_resolution(resolution_dropdown, aspect_ratio_dropdown)
    seed = int(random.randint(0, 2**32 - 1) if seed is None or seed < 0 else seed)

    image_start = Image.open(input_image_start).convert("RGB") if input_image_start else None
    image_end = Image.open(input_image_end).convert("RGB") if input_image_end else None

    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
    gen_kwargs = {"input_prompt": prompt, "image_start": image_start, "height": height, "width": width,
                  "frame_num": num_frames, "fps": 24.0, "seed": seed, "callback": None,
                  "VAE_tile_size": 256, "input_video_strength": 1.0, "denoising_strength": 1.0,
                  "guide_scale": float(guide_scale), "sampling_steps": int(num_steps),
                  "guide_phases": 2, "n_prompt": "", "enhance_prompt": False}
    if image_end:
        gen_kwargs["image_end"] = image_end

    result = ltx2_model.generate(**gen_kwargs)
    out_path = _get_next_output_path()
    if isinstance(result, dict):
        video_tensor, audio_data, audio_sr = result.get("x"), result.get("audio"), result.get("audio_sampling_rate", 24000)
    elif isinstance(result, tuple):
        video_tensor = result[0]; audio_data = result[1] if len(result) > 1 else None; audio_sr = result[2] if len(result) > 2 else 24000
    else:
        video_tensor, audio_data, audio_sr = result, None, 24000

    if video_tensor is None or not torch.is_tensor(video_tensor):
        return None, f"❌ No video tensor. Got: {type(video_tensor)}"

    video_tensor = video_tensor.cpu()
    video_for_save = video_tensor.unsqueeze(0).float() / 127.5 - 1.0
    save_video(tensor=video_for_save, save_file=out_path, fps=24.0, normalize=True, value_range=(-1, 1))

    if audio_data is not None:
        try:
            import tempfile, subprocess
            audio_tmp = tempfile.mktemp(suffix=".wav")
            if isinstance(audio_data, np.ndarray):
                audio_np = audio_data.T if audio_data.ndim == 2 and audio_data.shape[0] <= 2 else audio_data
                sf.write(audio_tmp, audio_np, int(audio_sr or 24000))
            elif torch.is_tensor(audio_data):
                import torchaudio
                cpu_audio = audio_data.cpu().float()
                torchaudio.save(audio_tmp, cpu_audio.squeeze(0) if cpu_audio.dim() == 3 else cpu_audio, int(audio_sr or 24000))
            subprocess.run(["ffmpeg", "-y", "-i", out_path, "-i", audio_tmp, "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-shortest", out_path.replace(".mp4", "_audio.mp4")], check=True, capture_output=True)
        except Exception:
            pass

    return out_path, f"✅ Done! Seed: {seed} | {width}x{height} | {num_frames} frames"

with gr.Blocks(css=".gradio-container{max-width:1000px;margin:auto!important}", theme=gr.themes.Soft(), title="LTX-2.3 Video Generator") as demo:
    gr.Markdown("# 🎬 LTX-2.3 22B Distilled 1.1 Q4 — Video Generator")
    with gr.Row():
        prompt = gr.Textbox(label="🎨 Prompt", lines=3, placeholder="A majestic eagle soaring over snowy peaks...")
    with gr.Accordion("🖼️ Image to Video (Optional)", open=False):
        with gr.Row():
            input_image_start = gr.Image(type="filepath", label="Start Frame")
            input_image_end = gr.Image(type="filepath", label="End Frame (Optional)")
    with gr.Row():
        seed = gr.Number(label="🎲 Seed (-1 = Random)", value=-1, precision=0)
        duration_dropdown = gr.Dropdown(choices=["2 Seconds (49 frames)", "3 Seconds (73 frames)", "5 Seconds (121 frames)",
                                        "8 Seconds (193 frames)", "10 Seconds (241 frames)", "15 Seconds (361 frames)"],
                                  value="5 Seconds (121 frames)", label="⏱️ Duration")
    with gr.Row():
        resolution_dropdown = gr.Dropdown(choices=["1080p", "720p", "540p", "480p"], value="720p", label="📐 Resolution")
        aspect_ratio_dropdown = gr.Dropdown(choices=["16:9 Landscape", "4:3 Standard", "1:1 Square", "3:4 Portrait", "9:16 Portrait"],
                                       value="16:9 Landscape", label="📏 Aspect Ratio")
    with gr.Row():
        guide_scale = gr.Slider(label="🎯 Guide Scale", minimum=1.0, maximum=8.0, step=0.5, value=3.0)
        num_steps = gr.Slider(label="⚡ Steps", minimum=2, maximum=8, step=1, value=8)
    with gr.Row():
        gen_btn = gr.Button("🎬 Generate", variant="primary")
    video_out = gr.Video(label="🎥 Output")
    status_out = gr.Textbox(label="Status", interactive=False)
    gen_btn.click(fn=Video_Generation, inputs=[prompt, input_image_start, input_image_end, seed,
                                             resolution_dropdown, aspect_ratio_dropdown, guide_scale, num_steps],
                  outputs=[video_out, status_out])

demo.launch(server_name="0.0.0.0", server_port=7860, share=True)

## Step 7: Run

In [ ]:
# For single video UI: run the cell above, then execute:
# !python -u run_ltx_t2v.py

# For headless bulk processing:
# !pip install -q ffmpeg-python  (if not already installed)
# !git clone https://github.com/kcblak/LTX-2.3-22B-bulk-1.1-Q4-Video-Generator-by-blak ltx-core 2>/dev/null || true
# %cd ltx-core
# !python main.py --init --project MyLTXProject
# !python main.py --run --headless

print("Uncomment the appropriate lines above and run to launch either headless or UI mode.")